In [1]:
from massive import RESTClient
import pandas as pd
import os
from tickers import NAS_100_TICKER
from tqdm import tqdm

client = RESTClient(api_key="SekkXsjUzVVSYEWYytpKll30sX_CCqlJ")

In [2]:
tickers = NAS_100_TICKER
save_path = "/Volumes/Elements/stock_market_data/nasdaq_100_2025_constituents/1_min/"
start_date = "2014-01-01"
end_date = "2025-12-01"
multiplier = 1
time_span = "minute"


for ticker in tqdm(tickers):
    # List Aggregates (Bars)
    results = []
    for a in client.list_aggs(ticker=ticker,
                            multiplier=multiplier,
                            timespan=time_span,
                            from_=start_date,
                            to=end_date,
                            adjusted=True,
                            limit=50000):

        results.append({'Open': a.open,
                        'High': a.high,
                        'Low': a.low,
                        'Close': a.close,
                        'Volume': a.volume,
                        'VWAP': a.vwap,
                        'Transactions': a.transactions,
                        #'OTC': a.otc,
                        'Timestamp': a.timestamp,
                        'Datetime': pd.to_datetime(a.timestamp, unit='ms')})

    df = pd.DataFrame(results)
    df = df.set_index('Datetime')

    desired_order = ['Close', 'High', 'Low', 'Open', 'Volume', 'VWAP', 'Transactions', 'Timestamp']
    df = df[desired_order]

    df.columns = pd.MultiIndex.from_product(
        [df.columns, [ticker]], 
        names=['Price', 'Ticker']
    )
    
    actual_start_date = df.index.min().strftime('%Y-%m-%d')
    actual_end_date = df.index.max().strftime('%Y-%m-%d')

    # save data
    file_name = f"{ticker}_1min_{actual_start_date}_to_{actual_end_date}.csv"
    file_path = os.path.join(save_path, file_name)
    df.to_csv(file_path)

100%|██████████| 102/102 [1:21:59<00:00, 48.23s/it]
